# Perseptron v2 - 04 Image-Only EfficientNet-B0 CNN Baseline

Bu notebook proposal'daki literal CNN/image-only baseline boslugunu kapatir. EfficientNet-B0 classifier head egitilir. Bu model kisisellestirilmis ana recommender degildir; image-only sanity baseline ve Grad-CAM icin kullanilir.

Cikti:

- `models/proposal_v2/image_only_effnet_cnn_fold{FOLD_ID}.pt`
- `reports/proposal_v2/proposal_v2_cnn_metrics.csv`

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

def find_source_project():
    candidates = [Path('/kaggle/working/perseptron_project'), Path.cwd(), Path('/kaggle/input/perseptron-project')]
    candidates += [path for path in Path('/kaggle/input').glob('*') if path.is_dir()]
    for path in candidates:
        if (path / 'src' / 'proposal_v2').exists():
            return path
    return Path.cwd()

SOURCE_PROJECT_DIR = find_source_project()
WORK_PROJECT_DIR = Path('/kaggle/working/perseptron_project_work') if Path('/kaggle').exists() else SOURCE_PROJECT_DIR
if SOURCE_PROJECT_DIR != WORK_PROJECT_DIR:
    shutil.copytree(SOURCE_PROJECT_DIR, WORK_PROJECT_DIR, dirs_exist_ok=True)

PROJECT_DIR = WORK_PROJECT_DIR
os.environ['PERSEPTRON_PROJECT_DIR'] = str(PROJECT_DIR)
os.environ['PYTHONPATH'] = str(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))
print('SOURCE_PROJECT_DIR =', SOURCE_PROJECT_DIR)
print('PROJECT_DIR =', PROJECT_DIR)

def restore_previous_outputs():
    if not Path('/kaggle/input').exists():
        return
    for input_root in Path('/kaggle/input').glob('*'):
        if input_root == SOURCE_PROJECT_DIR:
            continue
        for relative in ['reports/proposal_v2', 'models/proposal_v2']:
            source = input_root / relative
            target = PROJECT_DIR / relative
            if source.exists():
                target.mkdir(parents=True, exist_ok=True)
                shutil.copytree(source, target, dirs_exist_ok=True)
                print('Restored', source, '->', target)

restore_previous_outputs()

def run_module(module, *args):
    command = [sys.executable, '-m', module, *map(str, args)]
    print('RUN:', ' '.join(command))
    subprocess.run(command, cwd=PROJECT_DIR, check=True)


## Parametreler

CNN pahali oldugu icin full kosuda da temsil edici bir subset kullanilir. Proposal uyumu icin bu baseline'in varligi ve Grad-CAM uretebilmesi onemli.

In [ ]:
FAST_RUN = True
FOLD_ID = 0
MAX_TRAIN_POSITIVES = 1_000 if FAST_RUN else 20_000
MAX_VAL_POSITIVES = 300 if FAST_RUN else 5_000
EPOCHS = 1 if FAST_RUN else 2
BATCH_SIZE = 32 if FAST_RUN else 64
TRAIN_BACKBONE = False


## Egitim

`TRAIN_BACKBONE=False` iken EfficientNet feature layers dondurulur, classifier head egitilir. Sure yetmezse bu ayar final icin de korunabilir.

In [ ]:
args = [
    '--fold-id', FOLD_ID,
    '--max-train-positives', MAX_TRAIN_POSITIVES,
    '--max-val-positives', MAX_VAL_POSITIVES,
    '--epochs', EPOCHS,
    '--batch-size', BATCH_SIZE,
]
if TRAIN_BACKBONE:
    args.append('--train-backbone')
run_module('src.proposal_v2.cnn_baseline', *args)
